# 5. Model Development

This notebook develops and compares four models for predicting 30-day hospital readmission:

1. **Logistic Regression** — interpretable baseline.
2. **Random Forest** — captures non-linear relationships and feature interactions.
3. **Gradient Boosting** — improves predictions through sequential tree learning.
4. **Multilayer Perceptron (MLP)** — neural-network benchmark for more complex patterns.

The models provide a progression from a statistical baseline to more flexible machine-learning approaches.

Because 30-day readmission is an imbalanced outcome, **PR-AUC is the primary model-selection metric**. **Recall** is the key operational metric, reflecting the proportion of actual readmissions successfully identified.

ROC-AUC, precision, F1-score, and accuracy are reported as supporting metrics.

## 5.1 Import Libraries

Libraries are imported for data preparation, patient-aware splitting, preprocessing, model development, and performance evaluation.

In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    accuracy_score
)

## 5.2 Load the Modeling Dataset

The prepared modeling dataset is loaded for model development. The patient identifier is retained temporarily to support patient-aware data splitting but will not be used as a predictor.

In [3]:
df_model = pd.read_csv("../data/processed/modeling_data.csv")

print("Dataset shape:", df_model.shape)
df_model.head()

Dataset shape: (101763, 35)


,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,...,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,readmitted_30,diag_1_group,diag_2_group,diag_3_group
0,8222157,Caucasian,Female,[0-10),6,25,1,1,Other Specialty,41,...,No,No,No,No,No,No,0,Diabetes,Unknown,Unknown
1,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,59,...,No,No,Up,No,Ch,Yes,0,Other,Diabetes,Other
2,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,11,...,No,No,No,No,No,Yes,0,Other,Diabetes,Other
3,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,44,...,No,No,Up,No,Ch,Yes,0,Other,Diabetes,Circulatory
4,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,51,...,No,No,Steady,No,Ch,Yes,0,Neoplasms,Neoplasms,Diabetes


### 5.2.1 Restore Administrative Categorical Features

The administrative ID variables represent categories rather than numerical quantities. Since CSV files may reload these variables as numeric values, they are converted back to categorical format before modeling.

In [4]:
categorical_ids = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

df_model[categorical_ids] = df_model[categorical_ids].astype(str)

In [5]:
df_model[categorical_ids].dtypes

admission_type_id           str
discharge_disposition_id    str
admission_source_id         str
dtype: object

## 5.3 Patient-Aware Train-Test Split
Because patients may have multiple hospital encounters, the data is split by `patient_nbr` rather than by individual encounters. This prevents encounters from the same patient appearing in both the training and test sets.

Patient groups are retained for later cross-validation, while the patient identifier is removed before model training.

In [6]:
X = df_model.drop(columns="readmitted_30")
y = df_model["readmitted_30"]
groups = df_model["patient_nbr"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

groups_train = X_train["patient_nbr"].copy()
groups_test = X_test["patient_nbr"].copy()

### 5.3.1 Verify Patient Separation

The split is checked to confirm that no patient appears in both datasets.

In [7]:
print("Patient overlap:", len(set(groups_train) & set(groups_test)))

Patient overlap: 0


### 5.3.2 Finalize the Modeling Sets

The patient identifier is removed from the predictors after the groups have been preserved. The final partitions are then checked for size and readmission prevalence.

In [8]:
X_train.drop(columns="patient_nbr", inplace=True)
X_test.drop(columns="patient_nbr", inplace=True)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print(f"Train readmission rate: {y_train.mean():.2%}")
print(f"Test readmission rate: {y_test.mean():.2%}")

Train shape: (81670, 33)
Test shape: (20093, 33)
Train readmission rate: 11.22%
Test readmission rate: 10.92%


## 5.4 Modeling Preprocessor

The preprocessing strategy established in Notebook 04 is recreated for use within the model pipelines.

In [11]:
numerical_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

print(f"Numerical features: {len(numerical_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

Numerical features: 8
Categorical features: 25


## 5.5 Model Validation Strategy

Candidate models are compared using patient-aware cross-validation on the training set.`StratifiedGroupKFold` keeps encounters from the same patient within a single fold while also maintaining a similar readmission distribution across folds.

**PR-AUC** is the primary model-selection metric, with **recall** treated as the key operational metric. ROC-AUC, precision, F1-score, and accuracy are included as supporting measures.

The held-out test set is not used during model comparison and is reserved for final evaluation.

In [14]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "accuracy": "accuracy"
}

Five-fold patient-aware cross-validation will be applied consistently to each candidate model. Mean validation performance will then be compared across models, with PR-AUC serving as the primary criterion for model selection.